# Übung 01 – Klassifikation mit scikit-learn: Bank Marketing entlang von CRISP-DM

## Lernziel

Ich durchlaufe einen kompakten, aber vollständigen Klassifikationsworkflow: Problem präzisieren, Daten verstehen, Daten sauber aufteilen, Vorverarbeitung und Modell in einer Pipeline verbinden, Modelle per Cross-Validation vergleichen und die finale Leistung auf einer unberührten Testmenge beurteilen.

## Datensatz und Quelle

| Angabe | Quelle |
|---|---|
| Offizielle Dokumentation | https://archive.ics.uci.edu/dataset/222/bank+marketing |
| Direkter Download | https://archive.ics.uci.edu/static/public/222/bank+marketing.zip |
| DOI | https://doi.org/10.24432/C5K306 |
| Zitierform | Moro, S., Rita, P. & Cortez, P. (2014). *Bank Marketing* [Dataset]. UCI Machine Learning Repository. |
| Lizenz laut UCI | CC BY 4.0 |

## Fallfrage

**Kann die Bank auf Grundlage geeigneter, vor dem abschließenden Kontakt verfügbarer Merkmale einschätzen, ob ein Termingeldprodukt gezeichnet wird?**

Ich entferne `duration` bewusst. Die Gesprächsdauer ist erst nach Abschluss des entscheidenden Gesprächs bekannt und würde eine Prognose künstlich verbessern. Das ist ein typisches Beispiel für **Data Leakage**: Informationen gelangen ins Modell, die im echten Entscheidungsmoment nicht verfügbar wären. Als wissenschaftliche Referenz zur Formulierung, Erkennung und Vermeidung von Leakage dient Kaufman, Rosset, Perlich & Stitelman (2012), https://doi.org/10.1145/2382577.2382579.

In [ ]:
# Ich halte alle Imports am Anfang zusammen. Dadurch ist transparent,
# welche Bibliotheken mein Notebook benötigt und ich kann Fehler schneller eingrenzen.
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

# Die feste Zufallszahl macht zufällige Aufteilungen und Modellresultate reproduzierbar.
SEED = 42
np.random.seed(SEED)

# Diese Darstellung ist für die Analyse in Colab gut lesbar.
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', context='notebook')

DATA_DIR = Path('daten')
DATA_DIR.mkdir(exist_ok=True)


def download_and_extract_zip(url: str, label: str) -> Path:
    """Lädt ein offizielles ZIP-Archiv nur bei Bedarf herunter und entpackt es.

    Die Funktion ist absichtlich im Notebook sichtbar: Studierende sollen erkennen,
    dass die Datenquelle nicht manuell und nicht über einen lokalen Pfad bereitgestellt wird.
    """
    zip_path = DATA_DIR / f'{label}.zip'
    extract_dir = DATA_DIR / label

    if not zip_path.exists():
        print(f'Lade Daten von: {url}')
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        zip_path.write_bytes(response.content)
    else:
        print(f'Verwende vorhandenes Archiv: {zip_path}')

    if not extract_dir.exists():
        extract_dir.mkdir(parents=True)
        with ZipFile(zip_path) as archive:
            archive.extractall(extract_dir)

    return extract_dir

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay, classification_report, precision_recall_curve,
    PrecisionRecallDisplay, RocCurveDisplay, roc_auc_score, f1_score,
    precision_score, recall_score, make_scorer
)

## 1. Daten laden und die Archivstruktur nachvollziehen

Das UCI-Archiv enthält seinerseits ein Archiv `bank.zip`. Ich entpacke diese zweite Ebene ausdrücklich. In der Praxis sollte ich verschachtelte Archivstrukturen nie stillschweigend ignorieren, weil sonst versehentlich eine andere Datenversion verarbeitet werden kann.

In [ ]:
SOURCE_URL = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'
outer_dir = download_and_extract_zip(SOURCE_URL, 'bank_marketing')

nested_zip = outer_dir / 'bank.zip'
assert nested_zip.exists(), 'Erwartetes Unterarchiv bank.zip wurde nicht gefunden.'
inner_dir = outer_dir / 'bank'
if not inner_dir.exists():
    inner_dir.mkdir()
    with ZipFile(nested_zip) as archive:
        archive.extractall(inner_dir)

csv_path = inner_dir / 'bank-full.csv'
assert csv_path.exists(), f'Die erwartete Datei fehlt: {csv_path}'

df = pd.read_csv(csv_path, sep=';')
print(f'Datensatzform: {df.shape[0]:,} Zeilen und {df.shape[1]} Spalten')
df.head()

## 2. Data Understanding: Zielgröße, Verteilung und zulässige Merkmale

Vor dem Modellieren prüfe ich die Zielverteilung. Ich verwende eine stratifizierte Aufteilung, damit das Verhältnis von `yes` und `no` in Trainings- und Testdaten vergleichbar bleibt. Anschließend entferne ich `duration` vor jeder weiteren Verarbeitung.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='y', order=['no', 'yes'], color='#3b7ddd', ax=ax)
ax.set(title='Zielvariable: Zeichnung eines Termingeldprodukts', xlabel='Zeichnung', ylabel='Anzahl Kund:innen')
plt.show()

# Eine Prozentangabe macht die Klassenverteilung leichter vergleichbar.
target_share = df['y'].value_counts(normalize=True).rename_axis('y').mul(100).round(2)
display(target_share.to_frame('Anteil in Prozent'))

In [ ]:
# Das Merkmal duration wird aus fachlichen Gründen ausgeschlossen.
# Nicht jede statistisch hilfreiche Spalte ist auch ein zulässiges Prognosemerkmal.
leakage_features = ['duration']
X = df.drop(columns=['y'] + leakage_features)
y = (df['y'] == 'yes').astype(int)

print('Verwendete Merkmale:', list(X.columns))
print('Zielvariable: 1 = Termingeldprodukt gezeichnet, 0 = nicht gezeichnet')

## 3. Train-/Test-Split: Die Testmenge bleibt bis zum Schluss unberührt

Ich teile die Daten **vor** jeder lernenden Transformation. Die Testmenge soll später eine möglichst realistische Schätzung liefern, wie gut das finale Modell bei neuen Fällen arbeitet. Skalierung, Imputation und Kodierung werden ausschließlich aus den Trainingsdaten geschätzt; die `Pipeline` stellt dies technisch sicher.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)
print(f'Training: {X_train.shape[0]:,} Fälle | Test: {X_test.shape[0]:,} Fälle')
print(f'Positive Klasse im Training: {y_train.mean():.2%} | im Test: {y_test.mean():.2%}')

## 4. Eine Pipeline verbindet Datenvorbereitung und Modell

Ich unterscheide numerische und kategoriale Merkmale. Numerische Werte werden skaliert; kategoriale Werte werden per One-Hot-Encoding in modellierbare Indikatorspalten überführt. Beides geschieht innerhalb der Pipeline, damit Cross-Validation und Testbewertung keine Informationen aus Validierungs- oder Testdaten sehen.

In [ ]:
numeric_features = X_train.select_dtypes(include='number').columns.tolist()
categorical_features = X_train.select_dtypes(exclude='number').columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('numerisch', numeric_pipeline, numeric_features),
    ('kategorial', categorical_pipeline, categorical_features)
])

print(f'Numerische Merkmale ({len(numeric_features)}): {numeric_features}')
print(f'Kategoriale Merkmale ({len(categorical_features)}): {categorical_features}')

## 5. Baseline und zwei plausible Modelle vergleichen

Die Dummy-Baseline sagt immer die häufigste Klasse voraus und markiert damit die triviale Mindestleistung. Die logistische Regression ist transparent und liefert eine starke lineare Referenz. Der Random Forest kann nichtlineare Muster und Interaktionen abbilden. Ich entscheide nicht anhand eines einzigen Splits, sondern vergleiche alle Ansätze mit einer **stratifizierten 5-fachen Cross-Validation** auf den Trainingsdaten.

Die Baseline ist keine ernsthafte Modellalternative. Sie beantwortet die vorgelagerte Qualitätsfrage: **Schlägt mein Modell überhaupt eine einfache Regel?**

In [ ]:
models = {
    'Dummy: häufigste Klasse': DummyClassifier(strategy='most_frequent'),
    'Logistische Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED),
    'Random Forest': RandomForestClassifier(
        n_estimators=120,
        min_samples_leaf=5,
        class_weight='balanced',
        n_jobs=1,
        random_state=SEED
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {
    'roc_auc': 'roc_auc',
    'f1': make_scorer(f1_score, zero_division=0),
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0)
}

cv_rows = []
for name, model in models.items():
    pipeline = Pipeline(steps=[('vorbereitung', preprocessor), ('modell', model)])
    result = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append({
        'Modell': name,
        'ROC-AUC (Mittelwert)': result['test_roc_auc'].mean(),
        'F1 (Mittelwert)': result['test_f1'].mean(),
        'Precision (Mittelwert)': result['test_precision'].mean(),
        'Recall (Mittelwert)': result['test_recall'].mean()
    })

cv_results = pd.DataFrame(cv_rows).set_index('Modell').sort_values('ROC-AUC (Mittelwert)', ascending=False)
display(cv_results.style.format('{:.3f}'))

## 6. Finales Modell einmalig auf der Testmenge bewerten

Für die Lehrübung wähle ich das in der Cross-Validation stärkere **echte Modell** nach ROC-AUC; die Dummy-Baseline bleibt ausschließlich die Mindestanforderung. In einem realen Fall wäre die Wahl zusätzlich an Kosten, Risiko, Erklärbarkeit und Interventionskapazität auszurichten. Nach der Entscheidung trainiere ich das ausgewählte Modell auf allen Trainingsdaten und schaue **erst jetzt** auf die Testmenge.

In [ ]:
candidate_names = ['Logistische Regression', 'Random Forest']
best_model_name = cv_results.loc[candidate_names, 'ROC-AUC (Mittelwert)'].idxmax()
best_model = models[best_model_name]
final_pipeline = Pipeline(steps=[('vorbereitung', preprocessor), ('modell', best_model)])
final_pipeline.fit(X_train, y_train)

y_pred = final_pipeline.predict(X_test)
y_proba = final_pipeline.predict_proba(X_test)[:, 1]

print('Ausgewähltes Modell:', best_model_name)
print('Test-ROC-AUC:', round(roc_auc_score(y_test, y_proba), 3))
print('Test-F1:', round(f1_score(y_test, y_pred), 3))
print('\nKlassifikationsbericht:')
print(classification_report(y_test, y_pred, target_names=['nicht gezeichnet', 'gezeichnet']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['nein', 'ja'], cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix auf Testdaten')

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title('ROC-Kurve auf Testdaten')

PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=axes[2])
axes[2].set_title('Precision-Recall-Kurve auf Testdaten')
plt.tight_layout()
plt.show()

## 7. Transferfragen für die Prüfungsvorbereitung

1. Warum ist `duration` eine potenzielle Leakage-Quelle?
2. Warum ist die Dummy-Baseline trotz ihrer schwachen Leistung methodisch notwendig?
3. Warum werden Vorverarbeitung und Modell gemeinsam in einer Pipeline geschätzt?
5. Welchen Nachteil hätte es, die Testmenge mehrfach zu verwenden, um Parameter auszuwählen?
5. Welche Metrik würdest du priorisieren, wenn die Bank nur wenige Kund:innen ansprechen kann? Welche, wenn sie möglichst keine potenziell interessierten Kund:innen übersehen möchte?

> **Merksatz:** Ein guter Modellwert reicht nicht. Ich muss erklären können, welche Daten im Entscheidungsmoment verfügbar sind, wie ich validiert habe und welche Fehlerfolgen wichtig sind.